# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

/bin/bash: line 1: nvidia-smi: command not found


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [2]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [3]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.15


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [4]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)
MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    59 seconds ago            


### **CLIENTE DE PYTHON**

In [5]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [6]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
¡Excelente pregunta!

La RAM (Random Access Memory) y el almacenamiento (en general, se refiere a la memoria de estado sólido, SSD, o la memoria de estado sólido de clase rápida) son dos componentes clave de una computadora que desempeñan papeles diferentes, aunque están relacionados.

**La RAM (Random Access Memory)**

La RAM es una parte del sistema que se utiliza para almacenar datos y programas que se están ejecutando en la computadora. Es un tipo de memoria de acceso aleatorio, lo que significa que la computadora puede acceder directamente a cualquier elemento de la memoria sin tener que leerlos en orden. La RAM es muy rápida y eficiente, lo que la hace ideal para almacenar datos que se van a utilizar pronto.

La RAM tiene varias características clave:

*   Es temporal: los datos almacenados en la RAM se pierden cuando la computadora se apaga.
*   Es rápida: la RAM es muy rápida y eficiente.
*   Es Volátil

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [7]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)
# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste

Tokens del prompt: 43
Tokens de la respuesta: 537
Tokens totales: 580


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [8]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 208.73 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [9]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 208.73 s — 580 tokens
Modelo grande (llama3.1:8b): 98.68 s — 101 tokens (de los cuales 29.31 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM es una memoria temporal, es decir, la computadora la usa para ejecutar programas y procesar información, y cuando se apaga se pierde la información que estaba en ella. El almacenamiento se refiere al espacio de la computadora donde se almacenan los archivos y programas, incluso cuando la computadora no está encendida.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [19]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [20]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.15


In [21]:
# Descargar el modelo ligero de Llama e inicializar el cliente
import ollama
import time

# Descargar el modelo ligero
MODELO_LIGERO = "llama3.2:3b"

!ollama pull llama3.2:3b

# Inicializar cliente de Ollama
cliente = ollama.Client(host="http://localhost:11434")

modelo = MODELO_LIGERO

print("Cliente de Ollama inicializado correctamente con el modelo:", modelo)


Cliente de Ollama inicializado correctamente con el modelo: llama3.2:3b


**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
*  Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [22]:
# Definir la lista de preguntas
# Definir la lista de preguntas
preguntas = [
    "¿Qué es un transistor y para qué sirve?",
    "¿Cómo funciona un sensor infrarrojo?",
    "¿Qué diferencia existe entre una conexión en serie y una en paralelo?"
]

print("Lista de preguntas:")
for pregunta in preguntas:
    print("-", pregunta)

Lista de preguntas:
- ¿Qué es un transistor y para qué sirve?
- ¿Cómo funciona un sensor infrarrojo?
- ¿Qué diferencia existe entre una conexión en serie y una en paralelo?


In [23]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1
inicio = time.perf_counter()

respuesta = cliente.chat(
    model=modelo,
    messages=[
        {"role": "user", "content": preguntas[0]}
    ]
)

tiempo_respuesta = time.perf_counter() - inicio

resultado_1 = {
    "pregunta": preguntas[0],
    "respuesta": respuesta["message"]["content"],
    "tiempo": tiempo_respuesta,
    "tokens": respuesta.get("eval_count", 0)
}

print("Pregunta:", resultado_1["pregunta"])
print("Respuesta:", resultado_1["respuesta"])
print("Tiempo:", resultado_1["tiempo"], "segundos")
print("Tokens:", resultado_1["tokens"])

Pregunta: ¿Qué es un transistor y para qué sirve?
Respuesta: **¿Qué es un transistor?**

Un transistor es un componente electrónico que se utiliza para controlar el flujo de corriente eléctrica en un circuito. Fue inventado por John Bardeen, Walter Brattain y William Shockley en 1947 y revolucionó la industria electrónica.

**Cómo funciona un transistor**

Un transistor se compone de tres capas de material semiconductor: un emisor, un colector y un base. La base está conectada a la fuente de voltaje y se utiliza para controlar el flujo de corriente entre el emisor y el colector.

Cuando una corriente eléctrica fluye a través de la base, se crea una diferencia de potencial entre el emisor y el colector. Esto permite que la corriente fluya desde el emisor hacia el colector. El transistor puede actuar como un amplificador, un regulador de voltaje o un interruptor.

**Para qué sirve un transistor**

Los transistores tienen una amplia variedad de aplicaciones en la industria electrónica, in

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [24]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2
inicio = time.perf_counter()

respuesta = cliente.chat(
    model=modelo,
    messages=[
        {"role": "user", "content": preguntas[1]}
    ]
)

tiempo_respuesta = time.perf_counter() - inicio

resultado_2 = {
    "pregunta": preguntas[1],
    "respuesta": respuesta["message"]["content"],
    "tiempo": tiempo_respuesta,
    "tokens": respuesta.get("eval_count", 0)
}

print(resultado_2)

{'pregunta': '¿Cómo funciona un sensor infrarrojo?', 'respuesta': '**¿Qué es un sensor infrarrojo?**\n\nUn sensor infrarrojo es un dispositivo que detecta la radiación infrarroja, que es una forma de onda electromagnética que se encuentra más allá de la longitud de la visible luz. Los sensores infrarrojos están diseñados para detectar la radiación infrarroja emitida por objetos caleidos o calientes, lo que les permite medir temperaturas o detectar cambios en la temperatura.\n\n**Funcionamiento de un sensor infrarrojo**\n\nEl funcionamiento de un sensor infrarrojo se basa en el principe de que la radiación infrarroja se absorbe y se refleja por la superficie de un objeto. El sensor infrarrojo utiliza un cristal o un material que emite radiación infrarroja cuando se calienta, o que absorbe radiación infrarroja cuando se enfría.\n\nAquí hay una explicación detallada de cómo funciona un sensor infrarrojo:\n\n1. **Emitción de radiación infrarroja**: El sensor infrarrojo emite una radiación 

In [25]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3
inicio = time.perf_counter()

respuesta = cliente.chat(
    model=modelo,
    messages=[
        {"role": "user", "content": preguntas[2]}
    ]
)

tiempo_respuesta = time.perf_counter() - inicio

resultado_3 = {
    "pregunta": preguntas[2],
    "respuesta": respuesta["message"]["content"],
    "tiempo": tiempo_respuesta,
    "tokens": respuesta.get("eval_count", 0)
}

print(resultado_3)

{'pregunta': '¿Qué diferencia existe entre una conexión en serie y una en paralelo?', 'respuesta': 'Excelente pregunta!\n\nLas conexiones en serie y en paralelo son dos formas fundamentales de unir componentes eléctricos en un circuito. A continuación, te explicaré las diferencias clave entre ellas:\n\n**Conexión en serie**\n\nEn una conexión en serie, los componentes eléctricos se unen uno después de otro, de modo que la corriente fluye solo en una dirección. La corriente se suministra desde un extremo y fluye a través de cada componente, hasta el otro extremo.\n\nLas características de una conexión en serie son:\n\n* La corriente se suministra desde un extremo y fluye en una dirección.\n* Si un componente falla, el circuito queda desconectado.\n* La potencia total en serie es igual a la suma de la potencia de cada componente individual.\n* La velocidad de respuesta de cada componente se ve afectada por los componentes que lo preceden.\n\n**Conexión en paralelo**\n\nEn una conexión en

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [26]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []

resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

print("Los tres resultados se agregaron correctamente")
print(resultados)

Los tres resultados se agregaron correctamente
[{'pregunta': '¿Qué es un transistor y para qué sirve?', 'respuesta': '**¿Qué es un transistor?**\n\nUn transistor es un componente electrónico que se utiliza para controlar el flujo de corriente eléctrica en un circuito. Fue inventado por John Bardeen, Walter Brattain y William Shockley en 1947 y revolucionó la industria electrónica.\n\n**Cómo funciona un transistor**\n\nUn transistor se compone de tres capas de material semiconductor: un emisor, un colector y un base. La base está conectada a la fuente de voltaje y se utiliza para controlar el flujo de corriente entre el emisor y el colector.\n\nCuando una corriente eléctrica fluye a través de la base, se crea una diferencia de potencial entre el emisor y el colector. Esto permite que la corriente fluya desde el emisor hacia el colector. El transistor puede actuar como un amplificador, un regulador de voltaje o un interruptor.\n\n**Para qué sirve un transistor**\n\nLos transistores tiene

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [27]:
# Mostrar la tabla final de resultados
import pandas as pd

tabla = pd.DataFrame(resultados)

display(tabla)

print("\nCONCLUSIÓN:")
print("El modelo ligero Llama 3.2:3b respondió las tres preguntas.")
print("Se registraron las respuestas, los tiempos de ejecución y los tokens utilizados.")

,pregunta,respuesta,tiempo,tokens
0,¿Qué es un transistor y para qué sirve?,**¿Qué es un transistor?**\n\nUn transistor es...,214.849020,515
1,¿Cómo funciona un sensor infrarrojo?,**¿Qué es un sensor infrarrojo?**\n\nUn sensor...,270.084497,743
2,¿Qué diferencia existe entre una conexión en s...,Excelente pregunta!\n\nLas conexiones en serie...,181.265355,493



CONCLUSIÓN:
El modelo ligero Llama 3.2:3b respondió las tres preguntas.
Se registraron las respuestas, los tiempos de ejecución y los tokens utilizados.
